# SpriteForge trên Kaggle: mô tả nhân vật → sprite sheet pixel art

Kết hợp **Pixel Art Studio** (sinh nhân vật SD1.5 + hậu xử lý pixel) và **Sprite Sheep** (chuyển động bằng video WAN 2.2 TI2V 5B).

**Chuẩn bị (một lần, chi tiết trong `kaggle/README.md`):**
1. Kaggle → Settings: **verify số điện thoại** (bắt buộc để bật GPU và Internet).
2. Panel bên phải → *Session options*: **Accelerator = GPU T4** (T4 x2 cũng được), **Internet = On**.
3. **Add Input** → dataset `spriteforge-models` (chứa checkpoint SD1.5 pixel art). File WAN không có trong dataset thì notebook tự tải từ HuggingFace (~17GB).
4. Code SpriteForge: clone từ GitHub theo `SPRITEFORGE_REPO` + `SPRITEFORGE_BRANCH` ở cell 1. Repo private thì thêm secret `GITHUB_TOKEN` (*Add-ons → Secrets*).

**Chạy:** sửa nội dung `jobs.json` ở cell 5 → **Run All**, hoặc *Save Version → Save & Run All (Commit)* để chạy nền, không cần mở trình duyệt.
Kết quả: `output.zip` trong panel **Output**.

In [ ]:
# ==== 1. CẤU HÌNH ====
SPRITEFORGE_REPO = "https://github.com/neiknart2033/Text-To-Sprite-Sheet"  # bỏ qua nếu có dataset spriteforge-code
SPRITEFORGE_BRANCH = "v1"
# Repo private: Add-ons → Secrets → thêm GITHUB_TOKEN (fine-grained token, Contents: Read-only) và bật cho notebook này
COMFY_REF = ""           # để trống = ComfyUI mới nhất; điền tag/commit nếu muốn cố định phiên bản
RUN_BENCHMARK = False    # True = chạy thử 1 action (cell 4) để đo tốc độ WAN trên GPU này

WORK = "/tmp/spriteforge"            # ComfyUI + model (không lẫn vào Output)
OUTPUT = "/kaggle/working/output"    # kết quả (tải về từ panel Output)

In [ ]:
# ==== 2. CÀI ĐẶT: code SpriteForge + ComfyUI + model (~5–10 phút, 1 lần mỗi session) ====
import base64, glob, os, shutil, subprocess, sys

if "WORK" not in globals():
    raise RuntimeError("Chưa chạy cell 1 (CẤU HÌNH). Mỗi lần session khởi động lại "
                       "(VD sau khi đổi GPU/Internet) phải chạy lại từ cell 1.")

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

def git(*args, cwd=None):
    """git có token GITHUB_TOKEN (nếu có secret) qua header, không lộ token ra output."""
    auth = []
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        auth = ["-c", "http.extraHeader=Authorization: Basic "
                + base64.b64encode(f"x-access-token:{token}".encode()).decode()]
    except Exception:
        pass  # không có secret -> repo phải public
    r = subprocess.run(["git", *auth, *args], cwd=cwd, capture_output=True, text=True,
                       env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if r.returncode:
        raise RuntimeError(f"git {args[0]} lỗi:\n{r.stderr.strip()}\n"
                           "→ Repo private: thêm secret GITHUB_TOKEN. Sai nhánh: sửa SPRITEFORGE_BRANCH.")
    return r.stdout.strip()

os.makedirs(WORK, exist_ok=True)

# --- code SpriteForge: dataset "spriteforge-code" nếu có, không thì clone/pull từ GitHub
APP = f"{WORK}/app"
found = glob.glob("/kaggle/input/**/forge/pipelines.py", recursive=True)
if found:
    if not os.path.exists(APP):
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), APP)
    print("Code lấy từ dataset:", os.path.dirname(os.path.dirname(found[0])))
else:
    if os.path.exists(f"{APP}/.git"):
        git("pull", "--ff-only", cwd=APP)  # chạy lại cell = lấy code mới nhất
    else:
        git("clone", "--depth", "1", "--branch", SPRITEFORGE_BRANCH, SPRITEFORGE_REPO, APP)
    print("Code GitHub:", git("log", "-1", "--format=%h %s (%cr)", cwd=APP))

# --- ComfyUI
COMFY = f"{WORK}/ComfyUI"
if not os.path.exists(COMFY):
    if COMFY_REF:
        sh(f"git clone https://github.com/comfyanonymous/ComfyUI {COMFY} && cd {COMFY} && git checkout {COMFY_REF}")
    else:
        sh(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFY}")
sh(f"pip install -q -r {COMFY}/requirements.txt")

# --- model: gom về một thư mục bằng symlink (không copy), thiếu WAN thì tải từ HuggingFace
WAN_REPO = "Comfy-Org/Wan_2.2_ComfyUI_Repackaged"
WAN_FILES = {
    "diffusion_models": "split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors",
    "text_encoders": "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    "vae": "split_files/vae/wan2.2_vae.safetensors",
}
MODELS = f"{WORK}/models"
for folder in ("checkpoints", *WAN_FILES):
    os.makedirs(f"{MODELS}/{folder}", exist_ok=True)

wan_names = {os.path.basename(p) for p in WAN_FILES.values()}
inputs = glob.glob("/kaggle/input/**/*.safetensors", recursive=True) + glob.glob("/kaggle/input/**/*.ckpt", recursive=True)
for f in inputs:  # mọi checkpoint trong dataset (trừ file WAN) -> checkpoints/
    name = os.path.basename(f)
    dst = f"{MODELS}/checkpoints/{name}"
    if name not in wan_names and not os.path.exists(dst):
        os.symlink(f, dst)

for folder, path in WAN_FILES.items():
    name = os.path.basename(path)
    dst = f"{MODELS}/{folder}/{name}"
    if os.path.exists(dst):
        continue
    in_dataset = [f for f in inputs if os.path.basename(f) == name]
    if in_dataset:
        os.symlink(in_dataset[0], dst)
        print("WAN từ dataset:", name)
    else:
        from huggingface_hub import hf_hub_download
        print("Tải từ HuggingFace:", name, "...")
        shutil.move(hf_hub_download(WAN_REPO, path, local_dir=f"{WORK}/hf"), dst)

with open(f"{COMFY}/extra_model_paths.yaml", "w") as f:
    f.write(f"spriteforge:\n  base_path: {MODELS}\n  checkpoints: checkpoints\n"
            "  diffusion_models: diffusion_models\n  text_encoders: text_encoders\n  vae: vae\n")

for folder in sorted(os.listdir(MODELS)):
    print(f"{folder:17s}", os.listdir(f"{MODELS}/{folder}"))
assert os.listdir(f"{MODELS}/checkpoints"), "CHƯA có checkpoint SD1.5: Add Input dataset spriteforge-models!"
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")

In [ ]:
# ==== 3. KHỞI ĐỘNG ComfyUI (chạy nền) ====
import time, urllib.request
assert "COMFY" in globals(), "Chạy cell 1 và cell 2 trước"

def comfy_up():
    try:
        return urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=3).status == 200
    except Exception:
        return False

if not comfy_up():
    comfy_log = open(f"{WORK}/comfy.log", "w")
    comfy_proc = subprocess.Popen(
        [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--preview-method", "none"],
        cwd=COMFY, stdout=comfy_log, stderr=subprocess.STDOUT)
    for _ in range(150):  # tối đa ~5 phút
        if comfy_up() or comfy_proc.poll() is not None:
            break
        time.sleep(2)
if comfy_up():
    print("ComfyUI sẵn sàng")
else:
    print(open(f"{WORK}/comfy.log").read()[-4000:])
    raise RuntimeError("ComfyUI không khởi động được, xem log ở trên")

In [ ]:
# ==== 4. (TÙY CHỌN) BENCHMARK: 1 nhân vật + 1 action, in thời gian từng bước ====
assert "APP" in globals(), "Chạy cell 1, 2, 3 trước"
if RUN_BENCHMARK:
    !python {APP}/kaggle/run_batch.py --jobs {APP}/kaggle/jobs.bench.json --out /kaggle/working/bench

In [ ]:
%%writefile /kaggle/working/jobs.json
{
  "comfy_url": "http://127.0.0.1:8188",
  "checkpoint": "auto",
  "seed": -1,
  "steps": 30,
  "cfg_scale": 7,
  "frame_size": 128,
  "char_height": 64,
  "palette": 32,
  "fps": 10,
  "view": "side",
  "bg_color": "auto",
  "preview_scale": 3,
  "wan": {
    "steps": 20,
    "duration": 2.0
  },
  "jobs": [
    {
      "name": "hac_ronin",
      "mode": "prompt",
      "prompt": "dark samurai in black-indigo armor, oni mask with glowing red eyes, tattered black cloak, two katanas at the waist",
      "character_image": null,
      "actions": [
        {"name": "idle",   "prompt": "standing still in a ready stance, calm breathing, cloak moving slightly", "frames": 8},
        {"name": "walk",   "prompt": "walking in place, steady walk cycle, legs and arms swinging", "frames": 8, "align": "track"},
        {"name": "run",    "prompt": "running in place, fast run cycle, leaning forward", "frames": 8, "align": "track"},
        {"name": "attack", "prompt": "draws a katana and slashes horizontally, then returns to stance", "frames": 8},
        {"name": "heavy",  "prompt": "raises the katana overhead and strikes down hard", "frames": 8},
        {"name": "hurt",   "prompt": "recoils backward from a hit, then recovers", "frames": 6, "loop": false},
        {"name": "jump",   "prompt": "crouches, jumps straight up, then lands", "frames": 6, "loop": false},
        {"name": "death",  "prompt": "staggers, drops to the knees and collapses to the ground", "frames": 8, "loop": false, "duration": 3.0}
      ]
    }
  ]
}

In [ ]:
# ==== 6. CHẠY BATCH (chạy lại cell này = tiếp tục từ action chưa xong) ====
assert "APP" in globals(), "Chạy cell 1, 2, 3 trước"
!python {APP}/kaggle/run_batch.py --jobs /kaggle/working/jobs.json --out {OUTPUT}

In [ ]:
# ==== 7. ĐÓNG GÓI: bấm link "output.zip" hiện ra bên dưới để tải về ====
# (Kaggle không cho tải cả thư mục, chỉ tải từng file -> gom thành 1 file zip)
!cd /kaggle/working && rm -f output.zip && zip -qr output.zip output && ls -lh output.zip
from IPython.display import FileLink, display
os.chdir("/kaggle/working")
display(FileLink("output.zip", result_html_prefix="Tải kết quả: "))

In [ ]:
# ==== 8. (TÙY CHỌN) XEM NHANH các preview.gif ngay trong notebook ====
from IPython.display import HTML, display
gifs = sorted(glob.glob(f"{OUTPUT}/*/*/preview.gif"))
display(HTML("".join(
    f'<figure style="display:inline-block;margin:6px;text-align:center">'
    f'<img src="{os.path.relpath(g, "/kaggle/working")}" style="image-rendering:pixelated;background:#444">'
    f'<figcaption>{os.path.relpath(os.path.dirname(g), OUTPUT)}</figcaption></figure>' for g in gifs)))